# Lecture 3 — Policies, Timing, Populations
## 第三讲 —— 策略、时序、群体

**Computational Methods for Heterogeneous-Agent Macro**
**异质性主体宏观的计算方法**
Jeffrey Sun

Extend the L2 solver with: (i) a policy function via `argmax`,
(ii) a stochastic income process applied stage-by-stage,
(iii) forward simulation of a single household and of a whole population.

在 L2 的求解器之上扩展三件事：(i) 通过 `argmax` 得到策略函数；
(ii) 把随机收入过程按阶段引入；(iii) 单户与整个群体的前向模拟。

Steps:
1. Carry over `bellman_operator` and `solve_vfi` from L2 unchanged.
2. Add `policy_function(V, params)` — same as `bellman_operator` but with `argmax`.
3. Make timing explicit: each period decomposes into three stages —
   income shock $w_t \sim \Pi(\cdot \mid w_{t-1})$, income
   $b_t = R\,a_{t-1} + y_{w_t}$, consumption–saving $a_t = b_t - c^\star(b_t)$.
4. Simulate one household forward with `simulate_one`.
5. View a population as a vector on the wealth grid;
   read off cross-sectional welfare from $V$ for free.
6. Simulate a population forward with `simulate_population` —
   Stage 1 is a single matrix multiplication $\lambda \mapsto \lambda\,\Pi$.

步骤：
1. 把 L2 的 `bellman_operator` 与 `solve_vfi` 原样搬过来。
2. 加 `policy_function(V, params)`——和 `bellman_operator` 一样，只是把 `max` 换成 `argmax`。
3. 把时序写清楚：一期分三阶段——收入冲击
   $w_t \sim \Pi(\cdot \mid w_{t-1})$、收入
   $b_t = R\,a_{t-1} + y_{w_t}$、消费–储蓄 $a_t = b_t - c^\star(b_t)$。
4. 用 `simulate_one` 前推单户。
5. 把群体视为财富网格上的一个向量；从 $V$ 直接读出横截面福利。
6. 用 `simulate_population` 前推分布——第一阶段就是一次矩阵乘法
   $\lambda \mapsto \lambda\,\Pi$。

Notes:
- The agent's optimization still uses scalar income $y$ (same as L2). Stochastic income enters only at the simulation level. The fully self-consistent version (the agent prices in the income process) is L4.
- Code is written for clarity, not speed.

说明：
- 求解端的收入仍是标量 $y$（与 L2 相同）。随机收入只出现在模拟阶段。让家庭自己把收入过程内化的自洽版本，留到 L4。
- 代码以清晰为先，不求快。


### Environment
### 运行环境


In [ ]:
using Pkg
Pkg.activate("..")
Pkg.instantiate()
using Plots, Random, LinearAlgebra


# Core Code
# 核心代码


### Utility and Log Grid (carried from L2)
### 效用函数与对数网格（沿用 L2）

Identical to L2. The log grid clusters wealth points near the
borrowing constraint, where the policy is most nonlinear.

与 L2 完全相同。对数等距网格把更多点放在借贷约束附近——策略函数在那里最非线性。


In [ ]:
u(c) = c <= 0 ? -Inf : log(c)
loggrid(lo, hi, N) = exp.(range(log(lo), log(hi); length = N))


### Bellman Operator (carried from L2)
### 贝尔曼算子（沿用 L2）

Given a value function (guess) $V$, for each $b$ compute
$$
TV(b) = \max_{b'} \; u(b - b^{\mathrm{end}}) + \beta\, V(b'),
\qquad b^{\mathrm{end}} = \frac{b' - y}{R}.
$$

给定一个价值函数（猜测） $V$，对每个 $b$ 计算
$$
TV(b) = \max_{b'} \; u(b - b^{\mathrm{end}}) + \beta\, V(b'),
\qquad b^{\mathrm{end}} = \frac{b' - y}{R}.
$$


In [ ]:
"""
Apply Bellman operator V |-> TV, when V is a vector over the wealth grid and there are no income shocks.

作用一次贝尔曼算子 V |-> TV；V 是定义在财富网格上的向量，且无收入冲击。
"""
function bellman_operator(V, params)
    (; β, R, w, b_grid) = params

    # end-of-period wealth needed to reach each b' on the grid
    # 到达各个下期财富 b' 所需的期末财富
    b_end = (b_grid' .- w) ./ R

    return maximum(u.(b_grid .- b_end) .+ β .* V'; dims = 2)
end


### Value Function Iteration (carried from L2)
### 价值函数迭代（沿用 L2）

Iterate $V \mapsto \mathcal{T}V$ until the max-abs change drops below `tol`.

反复迭代 $V \mapsto \mathcal{T}V$，直到最大绝对变化小于 `tol`。


In [ ]:
function solve_vfi(params; tol = 1e-6, maxiter = 2000, verbosity = 0)
    V = zeros(size(params.b_grid))

    Δ, iters = Inf, 0
    while Δ >= tol
        TV = bellman_operator(V, params)
        Δ  = maximum(abs.(TV .- V))
        V  = TV
        iters += 1
        iters > maxiter && error("VFI did not converge in $maxiter iterations")
    end

    verbosity >= 1 && println("VFI converged in $iters iterations with error $Δ")
    return V
end


### Policy Function
### 策略函数

The policy is the same operator as the Bellman operator, with
`argmax` in place of `max`:
$$
c^\star(b) \;=\; \arg\max_{c \in [0, b]} \; u(c) + \beta\, V\!\big(R(b - c) + y\big).
$$

策略函数与贝尔曼算子是同一个算子，只是把 `max` 换成 `argmax`：
$$
c^\star(b) \;=\; \arg\max_{c \in [0, b]} \; u(c) + \beta\, V\!\big(R(b - c) + y\big).
$$


In [ ]:
"""
Compute the policy index vector `c_ind` from V. For each row of b_grid, returns
the column index in b_grid of the optimal next-period wealth b'.
Same operator as `bellman_operator`, with `argmax` instead of `max`.

由 V 算出策略下标向量 `c_ind`。对每个 b（行），返回最优下期财富 b' 在
b_grid 上的下标。与 `bellman_operator` 同一个算子，把 `max` 换成 `argmax`。
"""
function policy_function(V, params)
    (; β, R, w, b_grid) = params

    # exactly the same setup as `bellman_operator`
    # 与 `bellman_operator` 完全相同的准备
    b_end = (b_grid' .- w) ./ R

    # argmax over b' (columns) — pull out the column index for each row
    # 沿 b'（列）取 argmax；为每一行取出列下标
    idx = argmax(u.(b_grid .- b_end) .+ β .* V'; dims = 2)
    return vec(getindex.(idx, 2))::Vector{Int}
end


### Markov Income
### 马尔可夫收入

Income takes two values — employed ($y = 1.0$) or unemployed ($y = 0.3$) — with row-stochastic transition matrix
$$
\Pi \;=\; \begin{pmatrix} 0.95 & 0.05 \\ 0.50 & 0.50 \end{pmatrix}.
$$
Long-run employment share $\approx 0.91$; mean unemployment spell $= 2$ periods.

收入取两个值——就业 ($y = 1.0$) 与失业 ($y = 0.3$)——行随机转移矩阵
$$
\Pi \;=\; \begin{pmatrix} 0.95 & 0.05 \\ 0.50 & 0.50 \end{pmatrix}.
$$
长期就业占比约为 $0.91$，失业平均持续 $2$ 期。


In [ ]:
"""
Draw the next state of a Markov chain with row-stochastic Π, given current state i.

给定当前状态 i 与行随机转移矩阵 Π，抽取下一状态。
"""
sample_markov(Π, i) = findfirst(rand() .<= cumsum(Π[i, :]))


### A Population is a Vector
### 群体即向量

A *population* can be thought of as a *distribution* over state variables.

If $b$ is the only state variable, then a *population* is just a vector $\lambda$, with $\lambda_i$ = "number of people with wealth $b_i$."

If $(b, w)$ are the state variables, the population is a matrix $\lambda$, with $\lambda_{ij}$ = "number of people with wealth $b_i$ and income $w_j$."

*群体*可视为状态变量上的一个*分布*。

若 $b$ 是唯一的状态变量，则*群体*就是一个向量 $\lambda$，其中 $\lambda_i$ 表示“财富为 $b_i$ 的人数”。

若 $(b, w)$ 是状态变量，则群体是一个矩阵 $\lambda$，其中 $\lambda_{ij}$ 表示“财富为 $b_i$ 且收入为 $w_j$ 的人数”。


### Simulate a Population
### 群体模拟

We track each household by a pair of grid indices $(i_b, i_w)$. One period decomposes into three stages: Stage 1 advances $i_w$ via the Markov $\Pi$; Stages 2--3 send each $(i_b, i_w)$ cell to a single destination $(i_b', i_w)$ on the grid.

我们用一对网格下标 $(i_b, i_w)$ 跟踪每户。一期分三阶段：第一阶段按 $\Pi$ 推进 $i_w$；第二、第三阶段把每个 $(i_b, i_w)$ 单元确定地送到唯一的目标 $(i_b', i_w)$。


In [ ]:
"""
Snap a value `x` to the nearest grid index.

把数值 `x` 落到最近的网格下标。
"""
snap_idx(grid, x) = argmin(abs.(grid .- x))

"""
Roll one household forward T periods using the policy index vector `c_ind`.
Each period applies Stage 1 (Markov on income), then Stages 2--3 collapsed
into a single index lookup.

用策略下标向量 `c_ind` 将单户向前推 T 期。每期先做第一阶段（收入上的
马尔可夫），再用一次下标查找完成第二、第三阶段。
"""
function simulate_one(c_ind, params; T = 200, i_b_0 = 1, i_w_0 = 1)
    (; R, w, w_vals, Π, b_grid) = params
    i_b, i_w = i_b_0, i_w_0
    i_b_path, i_w_path = zeros(Int, T), zeros(Int, T)
    for t in 1:T
        # Stage 1: income shock
        # 第一阶段：收入冲击
        i_w = sample_markov(Π, i_w)

        # Stages 2 + 3: apply L02 policy + assemble cash-on-hand with the realized
        # income, then snap to the nearest grid index. (The L02 policy targets b'
        # assuming income = w; with realized w_vals[i_w] the actual b shifts.)
        # 第二、第三阶段：用 L02 策略 + 用实际收入装配现金，再落到最近的网格下标。
        # （L02 策略以 w 为收入选 b'；实际收入为 w_vals[i_w] 时 b 相应平移。）
        i_b = snap_idx(b_grid, b_grid[c_ind[i_b]] - w + w_vals[i_w])

        i_b_path[t], i_w_path[t] = i_b, i_w
    end
    return (; i_b_path, i_w_path)
end

"""
Roll a population distribution forward T periods.
Stage 1 is one matrix multiply (`λ * Π`). Stages 2--3 are a deterministic
scatter into the precomputed destination matrix `dest[i_b, i_w]`.

将群体分布向前推 T 期。第一阶段是一次矩阵乘法（`λ * Π`）。
第二、第三阶段是一次确定性的散布——按预先算好的目标矩阵
`dest[i_b, i_w]` 将质量散到目标格点。
"""
function simulate_population(c_ind, params, λ_0; T = 200)
    (; R, w, w_vals, Π, b_grid) = params

    # Pre-compute the destination wealth index for each (i_b, i_w_new) pair.
    # 预先算出每个 (i_b, i_w_new) 对应的目标财富下标。
    dest = snap_idx.(Ref(b_grid), b_grid[c_ind] .- w .+ w_vals')   # (N_b, N_w)

    λ_path = [copy(λ_0)]
    λ = copy(λ_0)
    for t in 1:T
        λ_mid = λ * Π                                          # Stage 1
        λ = scatter_into(λ_mid, dest)                          # Stages 2 + 3
        push!(λ_path, copy(λ))
    end
    return λ_path
end

"""
Scatter mass: for each source cell (i, j), add its mass into λ_new[dest[i, j], j].
The destination column matches the source column (income doesn't change in Stages 2--3).

散布质量：对每个源格 (i, j)，把其质量加到 λ_new[dest[i, j], j]。
目标列与源列相同（第二、第三阶段不改变收入）。
"""
function scatter_into(λ, dest)
    λ_new = zero(λ)
    for I in CartesianIndices(λ)
        λ_new[dest[I], I[2]] += λ[I]
    end
    return λ_new
end


## Demo
## 演示

Set up parameters, solve the value function, read off the policy, and run both simulators.

设好参数，解出价值函数，读出策略，然后跑两个模拟器。


In [ ]:
b_grid = loggrid(0.05, 40.0, 200)
w_vals = [1.0, 0.3]
Π      = [0.95 0.05;
          0.50 0.50]

params = (; β = 0.96, R = 1.04, w = 1.0,        # scalar w for the Bellman
            w_vals = w_vals, Π = Π,             # stochastic income for the simulation
            b_grid = b_grid)

V     = solve_vfi(params; verbosity = 1)
c_ind = policy_function(V, params)
;


### Plot $V$ and $c^\star$
### 画出 $V$ 与 $c^\star$


In [ ]:
# turn the index vector c_ind back into a consumption value for plotting:
# c = b - (b'_chosen - w) / R
# 将 c_ind 换算回消费值用于绘图：c = b - (b'_chosen - w) / R
c_pol = b_grid .- (b_grid[c_ind] .- params.w) ./ params.R

p1 = plot(b_grid, V; lw = 2, xlabel = "wealth b", ylabel = "V(b)",
          label = "V", legend = :bottomright)
p2 = plot(b_grid, c_pol; lw = 2, xlabel = "wealth b", ylabel = "c*(b)",
          label = "c*", legend = :bottomright)
plot!(p2, b_grid, b_grid; ls = :dash, lw = 1, label = "45°")
plot(p1, p2; layout = (1, 2), size = (900, 320))


### Simulate one household
### 模拟一户

Roll a single household forward over time. Wealth drifts up during employment spells and drops when income switches to unemployment.

把单户向前推若干期。就业期间财富缓慢攀升，失业时下落。


In [ ]:
Random.seed!(1)
# start at the grid point nearest to wealth 1.0, employed
# 初始位置：最接近财富 1.0 的网格点；就业状态
sim = simulate_one(c_ind, params; T = 300,
                   i_b_0 = argmin(abs.(b_grid .- 1.0)), i_w_0 = 1)

t_axis = 1:length(sim.i_b_path)
p1 = plot(t_axis, b_grid[sim.i_b_path]; lw = 1.5,
          xlabel = "t", ylabel = "wealth b_t",
          label = "b_t", legend = :topright)
p2 = plot(t_axis, [w_vals[w] for w in sim.i_w_path]; lw = 1.5,
          xlabel = "t", ylabel = "income w_t",
          label = "w_t", legend = :topright, seriestype = :steppost)
plot(p1, p2; layout = (2, 1), size = (820, 420))


### Simulate the population
### 群体模拟

Start everyone at the same wealth ($b \approx 1.0$) and the same income state (employed). Over time the population spreads out across the wealth grid, and after enough periods the wealth distribution stops changing.

让所有人都从同一个财富（$b \approx 1.0$）和同一个收入状态（就业）出发。随着时间推进，群体在财富网格上散开；经过足够多期后，财富分布就不再变化了。


In [ ]:
# start everyone at wealth ≈ 1.0, employed
# 让所有人从财富 ≈ 1.0、就业状态出发
λ_0 = zeros(length(b_grid), length(w_vals))
i_0 = argmin(abs.(b_grid .- 1.0))
λ_0[i_0, 1] = 1.0

λ_path = simulate_population(c_ind, params, λ_0; T = 200)
;


In [ ]:
# marginal wealth distribution at four snapshots
# 几个时刻的财富边际分布
function wealth_marginal(λ)
    return vec(sum(λ; dims = 2))
end

snapshots = [1, 5, 25, 200]
p = plot(xlabel = "wealth b", ylabel = "density", legend = :topright)
for t in snapshots
    plot!(p, b_grid, wealth_marginal(λ_path[t]); lw = 1.8, label = "t = $(t-1)")
end
plot(p; size = (820, 320), xscale = :log10)


### Cross-sectional aggregates from $V$ and $c^\star$
### 从 $V$、$c^\star$ 直接读出横截面总量

Aggregate welfare $\bar V$, consumption $\bar C$, wealth $\bar K$ at each $t$ — all inner products against the marginal wealth distribution.

总福利 $\bar V$、总消费 $\bar C$、总财富 $\bar K$——都是与边际财富分布的内积。


In [ ]:
# inner products against the marginal wealth distribution
# 与边际财富分布的内积
V_bar = [dot(V,      wealth_marginal(λ)) for λ in λ_path]
C_bar = [dot(c_pol,  wealth_marginal(λ)) for λ in λ_path]
K_bar = [dot(b_grid, wealth_marginal(λ)) for λ in λ_path]

t_axis = 0:length(λ_path)-1
p1 = plot(t_axis, V_bar; lw = 1.8, xlabel = "t", ylabel = "V̄_t", label = "V̄")
p2 = plot(t_axis, C_bar; lw = 1.8, xlabel = "t", ylabel = "C̄_t", label = "C̄")
p3 = plot(t_axis, K_bar; lw = 1.8, xlabel = "t", ylabel = "K̄_t", label = "K̄")
plot(p1, p2, p3; layout = (1, 3), size = (1000, 280))


# Step-by-Step Explanation
# 分步讲解


### Policy via `argmax`, broken down
### 用 `argmax` 提取策略：分步看


In [ ]:
# the Bellman operator's matrix of values, one entry per (b, b') pair
# 贝尔曼算子内部那张 (b, b') 矩阵
b_end = (b_grid' .- params.w) ./ params.R
M     = u.(b_grid .- b_end) .+ params.β .* V'
size(M)


In [ ]:
# `argmax` along columns: for each row (each b), which column (b') is best?
# 对每一行（每个 b），最大值落在哪一列（哪个 b'）？
idx = argmax(M; dims = 2)
idx[1:5]      # CartesianIndex(i, j*) for the first five b


In [ ]:
# pull out just the column index — this is the chosen b'-index
# 只取出列下标——即所选的 b' 下标
j_star = vec(getindex.(idx, 2))
j_star[1:5]


In [ ]:
# the chosen b'-index for each b is exactly what policy_function returns
# 对每个 b，所选 b' 的下标正是 policy_function 返回的内容
c_ind_check = vec(getindex.(idx, 2))
maximum(abs.(c_ind_check .- c_ind))   # should be 0


### Markov along one dimension is matrix multiplication
### 沿一维做马尔可夫就是矩阵乘法

Take just the income marginal $\lambda_w$ (length 2). One Markov step is $\lambda_w \cdot \Pi$. After many steps the income share stops changing — this is the long-run share of employed/unemployed implied by $\Pi$.

只看收入边际 $\lambda_w$（长度 2）。一次马尔可夫步就是 $\lambda_w \cdot \Pi$。迭代多次后，收入占比不再变化——这就是 $\Pi$ 所对应的长期就业/失业比例。


In [ ]:
# start: everyone employed
# 起点：所有人都就业
λ_w_0 = [1.0  0.0]                         # row vector

# one step
# 一步
λ_w_1 = λ_w_0 * Π

# iterate forward many times; the income split settles to its long-run value
# 反复迭代多次；收入占比趋于其长期值
λ_w = let λ = λ_w_0
    for t in 1:200
        λ = λ * Π
    end
    λ
end

(long_run_employed = λ_w[1], long_run_unemployed = λ_w[2])


### Trace one household through the three stages
### 跟踪一户走过三阶段

Pick a single $(i_b, i_w)$ cell, draw a Markov step, then apply Stages 2--3 (the cash-on-hand assembly + index lookup) to see where its mass ends up.

挑一个 $(i_b, i_w)$ 单元，先做一次马尔可夫抽样，再做第二、第三阶段（装配现金 + 下标查找），看看它的质量去到哪里。


In [ ]:
# trace one (i_b, i_w) cell through Stages 1, 2, 3
# 追踪一个 (i_b, i_w) 单元穿过第一、第二、第三阶段
i_b, i_w = 50, 1

# Stage 1: a Markov draw advances the income state
# 第一阶段：一次 Markov 抽样推进收入状态
Random.seed!(42)
i_w_new = sample_markov(params.Π, i_w)

# Stages 2 + 3: assemble cash-on-hand using the realized income, then snap
# 第二、第三阶段：用实际收入装配现金，再落到最近的网格下标
b_target = b_grid[c_ind[i_b]] - params.w + params.w_vals[i_w_new]
i_b_new = snap_idx(b_grid, b_target)

(i_b = i_b, i_w = i_w, i_w_new = i_w_new,
 b_target = b_target, i_b_new = i_b_new, b_new = b_grid[i_b_new])


### Backward iteration in stages
### 分阶段后向迭代

The three stages compose into one backward step on a 2-D value function
$V \in \mathbb{R}^{N_b \times N_w}$. Each stage maps one labelled $V$ to the next:

- **Stage 3 backward.** Grid-search $V^{\mathrm{end}} \mapsto V$.
- **Stage 2 backward.** Lookup $V \mapsto V^{\mathrm{start}}_{\mathrm{post}}$ at $R\,b^{\mathrm{end}} + w_vals[j]$.
- **Stage 1 backward.** Matrix-multiply $V^{\mathrm{start}}_{\mathrm{post}} \mapsto V^{\mathrm{start}}_{\mathrm{pre}}$ by $\Pi^\top$.

At the period boundary, $V^{\mathrm{end}} = \beta\, V^{\mathrm{start}}_{\mathrm{pre}}$.

三个阶段合在一起就是 2 维价值函数 $V \in \mathbb{R}^{N_b \times N_w}$ 的一次后向更新。每个阶段把一个 $V$ 映到下一个：

- **第三阶段（后向）。** 网格搜索 $V^{\mathrm{end}} \mapsto V$。
- **第二阶段（后向）。** 在 $R\,b^{\mathrm{end}} + w_vals[j]$ 上查表 $V \mapsto V^{\mathrm{start}}_{\mathrm{post}}$。
- **第一阶段（后向）。** 用 $\Pi^\top$ 做矩阵乘法 $V^{\mathrm{start}}_{\mathrm{post}} \mapsto V^{\mathrm{start}}_{\mathrm{pre}}$。

期界过渡：$V^{\mathrm{end}} = \beta\, V^{\mathrm{start}}_{\mathrm{pre}}$。


In [ ]:
"""
Stage 1 backward: V_start = V_pre_inc * Π'.

第一阶段（后向）：V_start = V_pre_inc * Π'。
"""
stage1_backward(V_pre_inc, Π) = V_pre_inc * Π'

"""
Stage 2 backward: V_pre_inc(b_end, w) = V(R*b_end + w, w), snapped to nearest grid.
Fully broadcast — no explicit loop.

第二阶段（后向）：V_pre_inc(b_end, w) = V(R*b_end + w, w)，落到最近的网格点。
全部用广播——没有显式循环。
"""
function stage2_backward(V, params)
    (; R, w_vals, b_grid) = params
    idx = snap_idx.(Ref(b_grid), R .* b_grid .+ w_vals')               # (N_b, N_w)
    return V[CartesianIndex.(idx, axes(idx, 2)')]                       # fancy indexing
end

"""
Stage 3 backward: grid search V(b, w) = max_{b_end} u(b - b_end) + V_end(b_end, w).
Returns (V, c_ind) where c_ind[i_b, i_w] is the optimal b_end-grid index.
Vectorized via a 3-D broadcast and `findmax(...; dims=2)`.

第三阶段（后向）：在 b_end 上做网格搜索，V(b, w) = max u(b - b_end) + V_end(b_end, w)。
返回 (V, c_ind)，其中 c_ind[i_b, i_w] 是最优 b_end 的网格下标。
用三维广播 + `findmax(...; dims=2)` 完成。
"""
function stage3_backward(V_end, params)
    b_grid = params.b_grid
    u_mat = u.(b_grid .- b_grid')                                       # (N_b, N_b_end)
    total = reshape(u_mat, length(b_grid), length(b_grid), 1) .+
            reshape(V_end, 1, size(V_end, 1), size(V_end, 2))           # (N_b, N_b_end, N_w)
    vals, idxs = findmax(total; dims = 2)
    return dropdims(vals; dims = 2),
           dropdims(getindex.(idxs, 2); dims = 2)::Matrix{Int}
end


Compose into one full backward step and iterate to a 2-D fixed point.
The resulting $V$ now varies with both wealth and income state — unlike
the L02 Bellman in this notebook, which fixes $y$ at its scalar mean.

把三个阶段合成一次完整的后向更新并迭代到 2 维不动点。这样得到的 $V$ 同时依赖财富与收入状态——与本笔记本上半段那个把 $y$ 固定为均值的 L02 贝尔曼不同。


In [ ]:
function bellman_staged(V_start_pre, params)
    (; β) = params
    V_end          = β .* V_start_pre                           # period boundary
    V, c_ind       = stage3_backward(V_end, params)             # Stage 3 backward
    V_pre_inc      = stage2_backward(V, params)                 # Stage 2 backward
    V_start_new    = stage1_backward(V_pre_inc, params.Π)       # Stage 1 backward
    return V_start_new, V, c_ind
end

# iterate the 2-D staged operator to convergence
# 迭代 2 维分阶段算子到收敛
V_start_2d, staged_iters = let V = zeros(length(params.b_grid), length(params.w_vals)), Δ = Inf, iters = 0
    while Δ >= 1e-6
        V_new, _, _ = bellman_staged(V, params)
        Δ = maximum(abs.(V_new .- V))
        V = V_new
        iters += 1
        iters > 2000 && error("staged VFI did not converge")
    end
    V, iters
end

V_2d, c_ind_2d = bellman_staged(V_start_2d, params)[2:3]
(staged_iters = staged_iters, V_2d_size = size(V_2d), V_emp_minus_unemp = V_2d[100, 1] - V_2d[100, 2])


### Forward iteration in stages
### 分阶段前向迭代

The three stages also compose into one *forward* step on a 2-D
distribution $\lambda \in \mathbb{R}^{N_b \times N_w}$:

- **Stage 1 forward.** $\lambda \mapsto \lambda\, \Pi$.
- **Stage 2 forward.** Each $(b^{\mathrm{end}}, w)$ cell moves to $(R\,b^{\mathrm{end}} + w_vals[j],\, w)$.
- **Stage 3 forward.** Each $(b, w)$ cell moves to $(b - c^\star(b, w),\, w)$.

Both wealth re-bins snap to the nearest grid point.

三个阶段也可以合成 2 维分布 $\lambda \in \mathbb{R}^{N_b \times N_w}$ 的一次*前向*更新：

- **第一阶段（前向）。** $\lambda \mapsto \lambda\, \Pi$。
- **第二阶段（前向）。** 每个 $(b^{\mathrm{end}}, w)$ 单元移到 $(R\,b^{\mathrm{end}} + w_vals[j],\, w)$。
- **第三阶段（前向）。** 每个 $(b, w)$ 单元移到 $(b - c^\star(b, w),\, w)$。

两次沿财富维的重排都落到最近的网格点。


In [ ]:
"""
Stage 1 forward: λ_post = λ_pre * Π.

第一阶段（前向）：λ_post = λ_pre * Π。
"""
stage1_forward(λ, Π) = λ * Π

"""
Stage 2 forward: each (i_b_end, i_w) cell moves to (snap(R*b_end + w), i_w).
Vectorize the destination index via broadcasting; one tight scatter.

第二阶段（前向）：每个 (i_b_end, i_w) 单元移到 (snap(R*b_end + w), i_w)。
目标下标用广播算出；一次紧凑散布。
"""
function stage2_forward(λ, params)
    (; R, w_vals, b_grid) = params
    dest = snap_idx.(Ref(b_grid), R .* b_grid .+ w_vals')               # (N_b, N_w)
    return scatter_into(λ, dest)
end

"""
Stage 3 forward: each (i_b, i_w) cell moves to (c_ind[i_b, i_w], i_w).
c_ind is exactly the destination index — direct scatter.

第三阶段（前向）：每个 (i_b, i_w) 单元移到 (c_ind[i_b, i_w], i_w)。
c_ind 本身就是目标下标——直接散布。
"""
stage3_forward(λ, c_ind, params) = scatter_into(λ, c_ind)

"""
One forward step of the composite operator T*:
λ → stage1 → stage2 → stage3.

复合算子 T* 的一次前向更新：λ → 第一阶段 → 第二阶段 → 第三阶段。
"""
function forward_step(λ, c_ind, params)
    λ = stage1_forward(λ, params.Π)
    λ = stage2_forward(λ, params)
    λ = stage3_forward(λ, c_ind, params)
    return λ
end


Iterate the forward step many times. The wealth distribution settles into a long-run shape; aggregates read off as inner products with $V$, $c^\star$, and $b$.

反复应用前向算子。财富分布最终趋于一个长期形态；总量通过与 $V$、$c^\star$、$b$ 的内积读出。


In [ ]:
# start everyone at the same wealth and income, then apply the forward step
# many times — the distribution stops changing once we've iterated long enough.
# 让所有人从同一个财富与收入出发，反复应用前向算子；
# 迭代足够多次后，分布就不再变化。
λ_long_run = let λ = zeros(length(params.b_grid), length(params.w_vals))
    i_0 = argmin(abs.(params.b_grid .- 1.0))
    λ[i_0, 1] = 1.0
    for t in 1:400
        λ = forward_step(λ, c_ind_2d, params)
    end
    λ
end

# aggregates in the long run
# 长期下的总量
λ_w = vec(sum(λ_long_run; dims = 1))         # income marginal
λ_b = vec(sum(λ_long_run; dims = 2))         # wealth marginal

(employed_share = λ_w[1],
 mean_wealth    = dot(params.b_grid, λ_b),
 mean_V         = sum(V_2d .* λ_long_run))
